# **Creación de un Banco de Datos**

## ***biblioteca sqlalchemy***
---
***Importar siguientes métodos:***
- **create_engine**: Crea el motor de datos para usar consultas sql, crea el entorno de trabajo.
- **MetaData**: Almacena información de los DataFrames que se van a crear (atributos, listas, diccionarios, etc).
- **Table**: Para trabajar con tablas.
- **inspect**: Para averiguar que tablas existen, atributos, elementos, etc.
- **text**: Convertir querys en tipo de dato de texto para manejar mejor las consultas.

---
*Este Banco de Datos local es el motor que soportará las tablas junto a sus elementos de forma integral*

In [1]:
import sqlalchemy
from sqlalchemy import create_engine, MetaData, Table, inspect, text

# **Creación del Banco de Datos local: SQLITE**

In [2]:
engine = create_engine ('sqlite:///:memory:') 
'''Objetivo especial (más que una variable) que actúa como puente de comunicación entre el entorno de python 
donde estarán los DataFrames y el sistema de base de datos (en este caso SQLite)'''

'Objetivo especial (más que una variable) que actúa como puente de comunicación entre el entorno de python \ndonde estarán los DataFrames y el sistema de base de datos (en este caso SQLite)'

verificamos tipo de archivo, a veces no son utf-8

In [3]:
import chardet
import pandas as pd


In [4]:
with open('clientes_banco.csv', 'rb') as file:
    print(chardet.detect(file.read()))

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


## **Insertar DataFrame "clientes" al gestor de Base de Datos local**

In [21]:
clientes = pd.read_csv ('clientes_banco.csv')
clientes.to_sql ('clientes', engine, index= False)

438463

### *Obtener lista de las tablas (DataFrames) del Banco de Datos*

In [22]:
inspeccion = inspect (engine)
print (inspeccion.get_table_names())

['clientes']


# **Leer consulta SQL**

## Examinar columnas
*Ver el nombre de las columnas existentes con una muestra aleatoria de 3 filas*

In [32]:
pd.read_sql (sql=text('SELECT * FROM clientes ORDER BY RANDOM () LIMIT 3'), con = engine.connect())

,ID_Cliente,Edad,Grado_estudio,Estado_civil,Tamaño_familia,Categoria_de_renta,Ocupacion,Años_empleado,Rendimiento_anual,Tiene_carro,Vivienda
0,6388914,54,Nivel superior,Casado,2,Empleado estatal,Otro,17,202500.0,0,Casa/Departamento propio
1,5956811,57,Nivel intermedio,Casado,2,Pensionista,Otro,0,81000.0,0,Casa/Departamento propio
2,5931570,38,Nivel intermedio,Casado,14,Business Partner,Equipo principal,7,337500.0,0,Casa/Departamento propio


In [26]:
empleados = pd.read_sql (sql=text('SELECT * FROM clientes WHERE Categoria_de_renta = "Empleado"'), con = engine.connect())
type (empleados)

pandas.core.frame.DataFrame

In [27]:
display (empleados.head())

,ID_Cliente,Edad,Grado_estudio,Estado_civil,Tamaño_familia,Categoria_de_renta,Ocupacion,Años_empleado,Rendimiento_anual,Tiene_carro,Vivienda
0,5008804,32,Nivel superior,Relación-estable,2,Empleado,Otro,12,427500.0,1,Departamento alquilado
1,5008805,32,Nivel superior,Relación-estable,2,Empleado,Otro,12,427500.0,1,Departamento alquilado
2,5008806,58,Nivel intermedio,Casado,2,Empleado,Seguridad,3,112500.0,1,Casa/Departamento propio
3,5008815,46,Nivel superior,Casado,2,Empleado,Contabilidad,2,270000.0,1,Casa/Departamento propio
4,5112956,46,Nivel superior,Casado,2,Empleado,Contabilidad,2,270000.0,1,Casa/Departamento propio


## Almacenar df de filtro "empleados" en el Banco de datos

In [28]:
empleados.to_sql ('empleados', engine, index = False)

226059

In [33]:
print (inspect(engine).get_table_names())

['clientes', 'empleados']


## Consultar algunas columnas

---
- ***Los siguientes dos códigos, leen la misma consulta***

In [34]:
pd.read_sql_table ('empleados', con= engine.connect(), columns= ['ID_Cliente', 'Edad', 'Ocupacion', 'Años_empleado'])

,ID_Cliente,Edad,Ocupacion,Años_empleado
0,5008804,32,Otro,12
1,5008805,32,Otro,12
2,5008806,58,Seguridad,3
3,5008815,46,Contabilidad,2
4,5112956,46,Contabilidad,2
...,...,...,...,...
226054,6837905,43,Otro,7
226055,6837906,43,Otro,7
226056,6839936,34,Construcción Civil,5
226057,6840222,43,Construcción Civil,8


In [42]:
pd.read_sql (sql= text 
            ('SELECT ID_Cliente, Edad, Ocupacion, Años_empleado ' \
            'FROM empleados ' \
            'LIMIT 5'), con= engine.connect())

,ID_Cliente,Edad,Ocupacion,Años_empleado
0,5008804,32,Otro,12
1,5008805,32,Otro,12
2,5008806,58,Seguridad,3
3,5008815,46,Contabilidad,2
4,5112956,46,Contabilidad,2


## **¿Cómo eliminar una fila de datos?**

### Importar biblioteca de información de errores

- from sqlalchemy.exc (.exc es ejecución) import SQLAlchemyError

In [43]:
from sqlalchemy.exc import SQLAlchemyError

#### Eliminar cliente ID: 5008804

---
***IMPORTANTE***
- **Cuando se quiere consultar datos utilizar:**
*pd.read_sql (sql= text ('código sql de consulta SELECT, etc'), con = engine.connect())*

- **Cuando se quiere modificar la DB:**
*engine.connect().execute (text ('código sql de modificación INSERT, UPDATE, DELETE, CREATE'))*

In [54]:
# Intentar eliminar los datos de ese ID de cliente, de lo contrario, que muestre exactamente el error del por qué no se puede.
try:
    r_set = engine.connect().execute(text ('DELETE FROM clientes WHERE ID_Cliente=5008804'))

except SQLAlchemyError as e:
    print (e) #Si sale error
else:
    print (f'Registros eliminados correctamente: {r_set.rowcount}')

Registros eliminados correctamente: 0


#### Verificar registro eliminado

In [55]:
pd.read_sql (sql= text
             ('SELECT * ' \
             'FROM clientes'), con = engine.connect())

,ID_Cliente,Edad,Grado_estudio,Estado_civil,Tamaño_familia,Categoria_de_renta,Ocupacion,Años_empleado,Rendimiento_anual,Tiene_carro,Vivienda
0,5008805,32,Nivel superior,Relación-estable,2,Empleado,Otro,12,427500.0,1,Departamento alquilado
1,5008806,58,Nivel intermedio,Casado,2,Empleado,Seguridad,3,112500.0,1,Casa/Departamento propio
2,5008808,52,Nivel superior,Soltero,1,Business Partner,Ventas,8,270000.0,0,Casa/Departamento propio
3,5008809,52,Nivel intermedio,Soltero,1,Business Partner,Ventas,8,270000.0,0,Casa/Departamento propio
4,5008810,52,Nivel intermedio,Soltero,1,Business Partner,Ventas,8,270000.0,0,Casa/Departamento propio
...,...,...,...,...,...,...,...,...,...,...,...
438457,6840104,62,Nivel intermedio,Divorciado,1,Pensionista,Otro,0,135000.0,0,Casa/Departamento propio
438458,6840222,43,Nivel intermedio,Soltero,1,Empleado,Construcción Civil,8,103500.0,0,Casa/Departamento propio
438459,6841878,22,Nivel superior,Soltero,1,Business Partner,Ventas,1,54000.0,0,Vive con los padres
438460,6842765,59,Nivel intermedio,Casado,2,Pensionista,Otro,0,72000.0,0,Casa/Departamento propio


### Actualización de un elemento (celda)

*Un cliente quiere que le actualicen su grado de estudio de Nivel intermedio a Nivel superior*
- ID cliente: 5008808

In [56]:
try:
    r_set = engine.connect().execute(text ('UPDATE clientes SET Grado_estudio="Nivel superior" WHERE ID_Cliente=5008808'))

except SQLAlchemyError as e:
    print (e)
else:
    print (f'Actualización realizada correctamente: {r_set.rowcount}')

Actualización realizada correctamente: 1


In [59]:
pd.read_sql (sql= text 
             ('SELECT ID_Cliente, Grado_estudio ' \
             'FROM clientes ' \
             'WHERE ID_Cliente=5008808'), con= engine.connect())

,ID_Cliente,Grado_estudio
0,5008808,Nivel superior
